# RadQA

## Install

In [1]:
!pip install -q openai
!pip install -q evaluate
!pip install -q transformers accelerate

In [2]:
import pandas as pd
import os
import openai
import concurrent.futures
from tqdm import tqdm
import time
import evaluate
import torch
# import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [3]:
from google.colab import drive
drive.mount('/content/drive')
# !ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Prompt

In [4]:
# # v1
# messages=[
#               {"role": "system", "content": "You are a radiologist. Your task is to determine whether a given context sentence from radiology reports contains the answer to a provided question. Please respond only with 'entailment' or 'not entailment' without any additional explanations. If you cannot determine, please respond with 'not entailment'."},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 1, redacted]\nQuestion: [RadQA demonstration question 1, redacted]"},
#               {"role": "assistant", "content": "entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 2, redacted]\nQuestion: [RadQA demonstration question 2, redacted]"},
#               {"role": "assistant", "content": "not entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: {}\nQuestion: {}".format(sentence, question)}
#           ]

# # v2
# messages=[
#               {"role": "system", "content": "You are a radiologist. Your task is to determine whether a given context sentence from radiology reports contains the answer to a provided question. Please respond *only* with 'entailment' or 'not entailment' without any additional explanations."},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 1, redacted]\nQuestion: [RadQA demonstration question 1, redacted]"},
#               {"role": "assistant", "content": "entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 2, redacted]\nQuestion: [RadQA demonstration question 2, redacted]"},
#               {"role": "assistant", "content": "not entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: {}\nQuestion: {}".format(sentence, question)}
#           ]

# # v3
# messages=[
#               {"role": "system", "content": "You are a radiologist. Your task is to determine whether a given context sentence from radiology reports contains the answer to a provided question. Please respond only with 'entailment' or 'not entailment' without any additional explanations."},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 1, redacted]\nQuestion: [RadQA demonstration question 1, redacted]"},
#               {"role": "assistant", "content": "entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 2, redacted]\nQuestion: [RadQA demonstration question 2, redacted]"},
#               {"role": "assistant", "content": "not entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 3, redacted]\nQuestion: [RadQA demonstration question 3, redacted]"},
#               {"role": "assistant", "content": "entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: {}\nQuestion: {}".format(sentence, question)}
#           ]

# # v5
# messages=[
#               {"role": "system", "content": "You are a radiologist. Your task is to determine whether a given context sentence from radiology reports contains the answer to a provided question. Please respond only with 'Yes' or 'No' without any additional explanations."},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 1, redacted]\nQuestion: [RadQA demonstration question 1, redacted]"},
#               {"role": "assistant", "content": "Yes"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 2, redacted]\nQuestion: [RadQA demonstration question 2, redacted]"},
#               {"role": "assistant", "content": "No"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 3, redacted]\nQuestion: [RadQA demonstration question 3, redacted]"},
#               {"role": "assistant", "content": "Yes"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: {}\nQuestion: {}".format(sentence, question)}
#           ]

## chatGPT

In [5]:
def chestGPT_summary(sentence, question):
  max_tries = 3
  for try_number in range(max_tries):
    try:
      response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
              {"role": "system", "content": "You are a radiologist. Your task is to determine whether a given context sentence from radiology reports contains the answer to a provided question. Please respond only with 'entailment' or 'not entailment' without any additional explanations."},
              {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 1, redacted]\nQuestion: [RadQA demonstration question 1, redacted]"},
              {"role": "assistant", "content": "entailment"},
              {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 2, redacted]\nQuestion: [RadQA demonstration question 2, redacted]"},
              {"role": "assistant", "content": "not entailment"},
              {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 3, redacted]\nQuestion: [RadQA demonstration question 3, redacted]"},
              {"role": "assistant", "content": "entailment"},
              {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: {}\nQuestion: {}".format(sentence, question)}
          ]
      )
      return response['choices'][0]['message']['content']

    # except Exception as e:
    #   if try_number == max_tries - 1:
    #     print('APIError')
    #     return '\n'
    #   else:
    #     time.sleep(0.1)
    
    except openai.error.APIError as e:
      if try_number == max_tries - 1:
        print('APIError')
        return '\n'
      else:
        time.sleep(0.1)
    except openai.error.Timeout as e:
      if try_number == max_tries - 1:
        print('Timeout')
        return '\n'
      else:
        time.sleep(0.1)
    except openai.error.APIConnectionError as e:
      if try_number == max_tries - 1:
        print('APIConnectionError')
        return '\n'
      else:
        time.sleep(0.1)
    except openai.error.RateLimitError as e:
      if try_number == max_tries - 1:
        print('RateLimitError')
        return '\n'
      else:
        time.sleep(0.1)

## Run

In [6]:
import time

def process_row(args):
    time.sleep(1)
    index, row = args
    corrected_text = chestGPT_summary(row['sentence'], row['question']).replace('not ', 'not_').replace('.', '')
    return index, corrected_text

# openai.api_key = '***REMOVED***' # gpt 3.5
# openai.api_key = '***REMOVED***' # cc
openai.api_key = os.environ['OPENAI_API_KEY']  # set the key in the environment (Colab: Secrets), never in the notebook

# input_file = "/content/drive/MyDrive/GPT_Data/R1/radqa_test_00.csv"
# input_file = "/content/drive/MyDrive/GPT_Data/R1/radqa_test_01.csv"
# input_file = "/content/drive/MyDrive/GPT_Data/R1/radqa_test_02.csv"

input_file = "/content/drive/MyDrive/GPT_Data/Random_sample/radqa_test_rand_01.csv"
# input_file = "/content/drive/MyDrive/GPT_Data/Random_sample/radqa_test_rand_02.csv"
# input_file = "/content/drive/MyDrive/GPT_Data/Random_sample/radqa_test_rand_03.csv"


df = pd.read_csv(input_file)

# parallel
# with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
with concurrent.futures.ThreadPoolExecutor() as executor:
    results = dict(tqdm(executor.map(process_row, df.iterrows()), total=len(df)))

# same order as the input
df['predict'] = df.index.map(results.get)

output_file = "/content/drive/MyDrive/GPT_Data/Random_sample/Output/radqa_test_rand_01_output_5.csv"

df.to_csv(output_file, index=False)

100%|██████████| 1787/1787 [15:21<00:00,  1.94it/s]


## Output

In [7]:
predict = pd.read_csv(output_file)
predict[['label', 'predict']].head(10)

,label,predict
0,not_entailment,not_entailment
1,not_entailment,not_entailment
2,not_entailment,not_entailment
3,entailment,entailment
4,not_entailment,not_entailment
5,not_entailment,not_entailment
6,not_entailment,not_entailment
7,not_entailment,not_entailment
8,not_entailment,not_entailment
9,not_entailment,not_entailment
